# NeuroTrain Lab — Notebook 3: Optimizers

**Topic:** how an optimizer actually uses the gradient (Notebook 2) to move the
weights, what role the learning rate plays, how SGD, Momentum, and Adam differ,
and why gradients sometimes "vanish" in deep networks.

> Notebook 3 of 4. You already know how to compute the loss's gradient. Now we
> use that gradient to **take steps** that reduce the error — and we'll see not
> all steps are equal.

## 🎯 What you'll learn in this notebook

1. What gradient descent does, step by step, over a loss function.
2. Why the learning rate is the most delicate decision in all of training.
3. What Momentum adds over plain SGD, and what makes Adam different (conceptually).
4. How to recognize a vanishing gradient in a deep network, and why it happens.
5. A quick diagnostic for "my network isn't learning" before touching code blindly.

**Mental map:** `gradient → descent step → learning rate → SGD → Momentum → Adam → vanishing gradients`

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_moons

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "breast_cancer_wisconsin.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from neurotrain.celebrations import celebrate
from neurotrain.visualization import plot_gradient_magnitude_by_layer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

print("NumPy:", np.__version__, "| TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)

## 0. Where we are

In Notebook 2 you learned to compute $\partial L / \partial w$ for every weight
in the network — the gradient tells you **which direction** the loss grows in.
What's missing is the rule that decides, gradient in hand, **how much and how** to
move each weight. That rule is the **optimizer**.

## 1. Gradient descent: the hiker in the fog

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 KEY CONCEPT — A hiker who can't see the valley</b><br><br>
Picture someone descending a mountain wrapped in thick fog. They can't see the
valley or the full map — they can only feel, underfoot, which way the ground
slopes down **at that exact spot**. They take a step in that direction, feel the
slope again, take another step, and so on.

That's exactly gradient descent: at each point we only know the **local**
gradient (the slope under our feet), not the full shape of the loss function. We
update like this:

$$w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$$

where $\eta$ (eta) is the **learning rate** — the size of the hiker's step.
</div>

In [ ]:
def f(w):
    """A toy 1D 'loss': a bowl with its minimum at w=3."""
    return (w - 3) ** 2 + 1


def gradient(w):
    return 2 * (w - 3)


def gradient_descent(w_start, lr, steps):
    trajectory = [w_start]
    w = w_start
    for _ in range(steps):
        w = w - lr * gradient(w)
        trajectory.append(w)
    return np.array(trajectory)


trajectory = gradient_descent(w_start=-2.0, lr=0.2, steps=15)
print("w positions:", trajectory.round(3))
print("Final loss:", round(f(trajectory[-1]), 4), "(true minimum is 1.0 at w=3)")

In [ ]:
w_curve = np.linspace(-3, 8, 200)
plt.figure(figsize=(7, 4.5))
plt.plot(w_curve, f(w_curve), color="#94A3B8", label="f(w) = (w-3)² + 1")
plt.plot(trajectory, f(trajectory), "o-", color="#7C3AED", label="descent steps")
for i in range(len(trajectory) - 1):
    plt.annotate(
        "", xy=(trajectory[i + 1], f(trajectory[i + 1])),
        xytext=(trajectory[i], f(trajectory[i])),
        arrowprops=dict(arrowstyle="->", color="#F97316", alpha=0.6),
    )
plt.scatter([3], [1], color="#22C55E", zorder=5, label="true minimum (w=3)")
plt.title("The hiker descending the loss bowl, step by step")
plt.xlabel("w")
plt.ylabel("f(w)")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

### ✏️ Exercise

Complete `gradient_descent_manual`: at each step, compute the gradient at the
current `w` and update it by subtracting `lr * gradient`. Test it with
`w_start=6.0, lr=0.3, steps=10` and confirm `w` gets close to 3.

In [ ]:
def gradient_descent_manual(w_start, lr, steps):
    w = w_start
    for _ in range(steps):
        g = gradient(w)
        w = w - ✏️✏️✏️
    return w


final_w = gradient_descent_manual(w_start=6.0, lr=0.3, steps=10)
print("Final w:", round(final_w, 4))

<details>
<summary><b>Show solution</b></summary>

```python
def gradient_descent_manual(w_start, lr, steps):
    w = w_start
    for _ in range(steps):
        g = gradient(w)
        w = w - lr * g
    return w


final_w = gradient_descent_manual(w_start=6.0, lr=0.3, steps=10)
print("Final w:", round(final_w, 4))
```

</details>

## 2. The effect of the learning rate

<div style="border-left:4px solid #2563EB; background:#EFF6FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>❓ LIKELY QUESTION — Why not always use a huge learning rate to get there faster?</b><br><br>
Because if the hiker takes steps that are too large, they can leap right over the
valley and land on the opposite slope — or even farther from where they started.
A small step is safe but slow; a large step is fast but can bounce forever, or
diverge outright. Let's see it with numbers.
</div>

In [ ]:
scenarios = {
    "lr too small (0.01)": gradient_descent(-2.0, lr=0.01, steps=40),
    "good lr (0.3)": gradient_descent(-2.0, lr=0.3, steps=40),
    "lr too large (1.05)": gradient_descent(-2.0, lr=1.05, steps=40),
}

plt.figure(figsize=(8, 4.5))
colors = ["#2563EB", "#22C55E", "#F97316"]
for (name, scenario_trajectory), color in zip(scenarios.items(), colors):
    plt.plot(scenario_trajectory, marker=".", label=name, color=color)
plt.axhline(3, color="#94A3B8", linestyle="--", linewidth=0.8, label="minimum (w=3)")
plt.title("Same starting point, three different learning rates")
plt.xlabel("step")
plt.ylabel("w")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

for name, scenario_trajectory in scenarios.items():
    print(f"{name}: w after 40 steps = {scenario_trajectory[-1]:.3f}")

With `lr=0.01` the hiker has barely moved after 40 steps: still far from the
minimum. With `lr=0.3` it converges smoothly. With `lr=1.05` every step sends it
**farther** than the last — it's diverging, not converging.

<div style="border-left:4px solid #F97316; background:#FFF7ED; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>⚠️ TYPICAL MISTAKE — A learning rate that's too high isn't always obvious at first glance</b><br><br>
In a real network, too high a learning rate sometimes produces a loss that
**drops a bit at first** and then starts oscillating or blows up to `NaN`. Don't
assume "if it started dropping, the learning rate is fine" — keep watching the
curve for several more epochs.
</div>

### ✏️ Exercise

Pick a fourth learning rate to try (any positive number other than 0.01, 0.3, or
1.05). **Before running the cell**, write in a comment what you think will happen
(smooth convergence / too slow / diverges). Then run it and check if you were right.

In [ ]:
# My prediction: ✏️✏️✏️ (write smooth convergence / too slow / diverges here)
my_lr = 0.6
my_trajectory = gradient_descent(-2.0, lr=my_lr, steps=40)
print("w after 40 steps:", round(my_trajectory[-1], 3))
plt.plot(my_trajectory, marker=".", color="#7C3AED")
plt.axhline(3, color="#94A3B8", linestyle="--")
plt.title(f"My learning rate = {my_lr}")
plt.show()

<details>
<summary><b>Show solution</b></summary>

```python
# My prediction: with lr=0.6 the step is larger than optimal but still within the
# range that converges (0 < lr < 1 for this parabola); I'd expect damped
# oscillation that still ends up close to the minimum.
my_lr = 0.6
my_trajectory = gradient_descent(-2.0, lr=my_lr, steps=40)
print("w after 40 steps:", round(my_trajectory[-1], 3))
plt.plot(my_trajectory, marker=".", color="#7C3AED")
plt.axhline(3, color="#94A3B8", linestyle="--")
plt.title(f"My learning rate = {my_lr}")
plt.show()
```

</details>

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🔥 Halfway there: from here it's a short hop to unravelling the mysteries of the artificial mind.</div>

## 3. SGD, Momentum, and Adam: the same MLP, three optimizers

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 KEY CONCEPT — Momentum: a ball that keeps some of its speed</b><br><br>
The hiker from Section 1 decides each step by looking **only** at the current
slope, as if starting fresh every time. **Momentum** is different: instead of a
hiker, picture a ball rolling downhill — it keeps part of the velocity it already
had. That lets it roll through small bumps without stalling out, and speed up
along stretches where the slope keeps pointing the same way.

**Adam** goes a step further: it adapts the step size **separately for each
parameter**, using accumulated statistics of recent gradients. You don't need to
memorize its formula — just know it combines the idea of Momentum with adaptive
per-parameter steps, which is why it tends to converge fast "out of the box"
without much manual learning-rate tuning.
</div>

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.25, random_state=RANDOM_STATE)

plt.figure(figsize=(4.5, 4))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="RdBu_r", edgecolor="white")
plt.title("make_moons (noisier than Notebook 1): the challenge for our 3 optimizers")
plt.show()

In [ ]:
def build_mlp():
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(2,)),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])


optimizers = {
    "SGD": tf.keras.optimizers.SGD(learning_rate=0.05),
    "SGD + Momentum": tf.keras.optimizers.SGD(learning_rate=0.05, momentum=0.9),
    "Adam": tf.keras.optimizers.Adam(learning_rate=0.05),
}

histories = {}
for name, optimizer in optimizers.items():
    model = build_mlp()
    model.compile(optimizer=optimizer, loss="binary_crossentropy")
    history = model.fit(X_moons, y_moons, epochs=50, verbose=0)
    histories[name] = history.history["loss"]
    print(f"{name:16s} -> initial loss {history.history['loss'][0]:.3f}, "
          f"final loss {history.history['loss'][-1]:.3f}")

In [ ]:
plt.figure(figsize=(7.5, 4.5))
optimizer_colors = {"SGD": "#F97316", "SGD + Momentum": "#2563EB", "Adam": "#22C55E"}
for name, losses in histories.items():
    plt.plot(losses, label=name, color=optimizer_colors[name])
plt.title("Same network, same data, same learning rate: only the optimizer changes")
plt.xlabel("epoch")
plt.ylabel("loss (binary cross-entropy)")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

With the same `learning_rate=0.05` for all three, you'll see plain SGD takes the
longest to bring the loss down, Momentum speeds that descent up, and Adam
typically converges faster and more steadily from the earliest epochs — without
touching any other hyperparameter.

<div style="border-left:4px solid #22C55E; background:#F0FDF4; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>📌 REMEMBER THIS — Adam isn't magic, it's a reasonable default</b><br><br>
Adam tends to work well "as is" on many problems, which is why it's such a
popular starting point. But "reasonable default" isn't "always optimal": for
specific problems, well-tuned SGD + Momentum sometimes generalizes better.
Choosing an optimizer is still a decision you validate, not a fixed rule.
</div>

## 4. Vanishing gradients: why depth isn't free

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 KEY CONCEPT — Multiplying many small numbers, many times</b><br><br>
In Notebook 2 you saw backpropagation uses the **chain rule**: the gradient at a
deep layer is computed by multiplying, layer by layer, the local derivatives of
every layer between it and the output. Sigmoid's derivative never exceeds **0.25**
(it peaks at $z=0$). Chain together 8-10 layers of Sigmoid and you're multiplying
8-10 numbers each at most 0.25 — the result shrinks to almost nothing very fast.
ReLU, by contrast, has derivative 1 (for active neurons) or 0, so it doesn't
crush the gradient the same way when chained.
</div>

In [ ]:
def build_deep_network(activation, depth=9):
    layers = [tf.keras.layers.Input(shape=(2,))]
    for _ in range(depth):
        layers.append(tf.keras.layers.Dense(16, activation=activation))
    layers.append(tf.keras.layers.Dense(1, activation="sigmoid"))
    return tf.keras.Sequential(layers)


def gradient_magnitudes(model, X, y):
    """Mean |gradient| of each Dense layer's kernel, output-to-input order."""
    X_t = tf.convert_to_tensor(X, dtype=tf.float32)
    y_t = tf.convert_to_tensor(y.reshape(-1, 1), dtype=tf.float32)
    with tf.GradientTape() as tape:
        prediction = model(X_t, training=True)
        loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_t, prediction))
    grads = tape.gradient(loss, model.trainable_weights)
    kernels = [g for w, g in zip(model.trainable_weights, grads) if "kernel" in w.name]
    magnitudes = [float(tf.reduce_mean(tf.abs(g))) for g in kernels]
    magnitudes.reverse()  # from the layer closest to the output toward the input
    return magnitudes


magnitudes_by_activation = {}
for activation in ["sigmoid", "relu"]:
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    network = build_deep_network(activation)
    magnitudes_by_activation[activation] = gradient_magnitudes(network, X_moons, y_moons)
    print(activation, [f"{m:.2e}" for m in magnitudes_by_activation[activation]])

In [ ]:
plot_gradient_magnitude_by_layer(magnitudes_by_activation, lang="en")
plt.show()

With this exact architecture (9 hidden layers of 16 neurons, seed 42), the Sigmoid
layer closest to the output has a mean gradient of **~4.0 × 10⁻²**, and the layer
closest to the input drops to **~1.6 × 10⁻⁸** — a fall of **more than 6 orders of
magnitude**. With ReLU, every layer stays in a similar range, between **~1.6 × 10⁻⁴
and ~9.4 × 10⁻⁴**, with no tendency to vanish. That layer closest to the input in
the Sigmoid network receives such a tiny gradient it essentially **stops learning**:
its weights barely change from one training pass to the next.

<div style="border-left:4px solid #2563EB; background:#EFF6FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>❓ LIKELY QUESTION — So should I never use Sigmoid in hidden layers?</b><br><br>
As a practical rule: use ReLU (or variants like Leaky ReLU) in the **hidden
layers** of deep networks, and reserve Sigmoid for the **output layer** in binary
classification, where there's only one such layer and the problem doesn't show
up. That's exactly what you've been doing since Notebook 1 — now you know why.
</div>

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🚀 The network is taking shape under your hands.</div>

## 5. 'Why isn't my network learning?' — quick diagnostic table

| Symptom | Likely cause | What to check |
|---|---|---|
| Loss is flat from epoch 1 | Learning rate too small / vanishing gradient / dead ReLU neurons | Raise the learning rate; check per-layer gradient magnitude (Section 4); check how many ReLU activations always output 0 |
| Loss explodes or becomes `NaN` | Learning rate too high / numerical instability | Lower the learning rate; check for unnormalized inputs or extreme logits |
| Training loss improves but validation loss gets worse | Overfitting (the model is memorizing, not generalizing) | Not covered here — this is the central topic of Notebook 4 |
| Loss drops very slowly but steadily | Learning rate somewhat low, or an optimizer with no momentum on tricky curvature | Try Momentum or Adam (Section 3) before touching the architecture |
| Loss oscillates without clearly dropping | Learning rate too high for that optimizer | Lower the learning rate; compare with Section 2 |

<div style="border-left:4px solid #22C55E; background:#F0FDF4; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>📌 REMEMBER THIS — Check the optimizer before changing the architecture</b><br><br>
When a network isn't learning, it's tempting to immediately add layers or
neurons. But many of the most common symptoms (flat loss, exploding loss,
painfully slow convergence) come down to the **learning rate** or the
**optimizer**, not network size. Check the cheap fixes first.
</div>

## 🎯 Self-assessment

Answer without looking back. You don't need perfect phrasing: explain the mechanism in your own words.

**1. What does Momentum add over plain gradient descent (SGD)?**

A. Nothing, they're mathematically identical
B. It keeps part of the 'velocity' from previous steps, smoothing and speeding up the trajectory
C. It completely removes the need to choose a learning rate
D. It only works if the network has a single layer

<details>
<summary><b>Show answer</b></summary>

**B.** Momentum accumulates a fraction of the previous step, like a rolling ball that keeps its speed, instead of deciding each step from the current slope alone.

</details>

**2. In a per-layer gradient-magnitude chart (like the one in Section 4), how do you recognize a vanishing-gradient problem?**

A. Every layer has a similar magnitude, with no drop across depth
B. The magnitude drops several orders of magnitude the farther you get from the output layer
C. The magnitude rises exponentially near the input
D. The chart is unrelated to the problem; you should look at the loss instead

<details>
<summary><b>Show answer</b></summary>

**B.** The telltale pattern is a steep (multiplicative, hence visible on a log scale) drop in gradient magnitude in the layers closest to the input.

</details>

**3. Per the diagnostic table in Section 5, what should you suspect first if training loss and validation loss diverge (validation gets worse)?**

A. A learning rate that's too high
B. Overfitting — a topic developed further in Notebook 4
C. A vanishing gradient
D. A typo in the loss function

<details>
<summary><b>Show answer</b></summary>

**B.** Train improving while validation worsens is the classic signature of overfitting, which this notebook only flags — it's covered in depth in Notebook 4.

</details>

**4. In the Section 2 experiment, what happened to the trajectory with `lr=1.05`?**

A. It converged faster than with `lr=0.3`
B. It stayed exactly at the starting point
C. Each step moved farther from the minimum than the last: it diverges
D. It reached the minimum but oscillated slightly around it

<details>
<summary><b>Show answer</b></summary>

**C.** With a learning rate above the stable range for this parabola, each update overshoots the minimum by a growing margin: the trajectory diverges instead of converging.

</details>

**5. What does Adam do differently from Momentum, conceptually (no formulas)?**

A. Adam doesn't use a learning rate at all
B. Adam adapts the step size for each parameter individually, using accumulated gradient statistics
C. Adam only works for Softmax classification
D. Adam is identical to SGD without momentum

<details>
<summary><b>Show answer</b></summary>

**B.** Adam combines the idea of Momentum with per-parameter adaptive steps, computed from accumulated statistics of recent gradients (roughly, mean and variance).

</details>

**6. Why do you think Adam became the default choice in industry? Explain in your own words.**

<details>
<summary><b>What a good answer should include</b></summary>

- Mentions it tends to converge fast with little manual hyperparameter tuning.
- Connects the idea of per-parameter adaptive steps to problems where different weights need different update scales.
- Acknowledges 'reasonable default' doesn't mean 'always best' — it's still a choice you validate.

</details>

**7. A teammate tells you: 'my network's loss has been flat since epoch 1'. Walk through, step by step, how you'd diagnose it.**

<details>
<summary><b>What a good answer should include</b></summary>

- Starts by checking the learning rate (too small?) before touching the architecture.
- Mentions checking per-layer gradient magnitude to rule out a vanishing gradient.
- Considers dead ReLU neurons or unnormalized input data as possibilities.
- Follows a reasoned order instead of changing things at random, leaning on the Section 5 diagnostic table.

</details>

In [ ]:
celebrate(
    "🎉 Congratulations! You finished Notebook 3: Optimizers 🎉",
    "You now know how an optimizer turns a gradient into a step, what each of "
    "SGD/Momentum/Adam brings to the table, and how to diagnose a network that "
    "isn't learning. In Notebook 4 we train for real: epochs, batches, and how to "
    "avoid overfitting.",
)